# 02 · Cure Models (Classification)

Predicts the probability that a defaulted loan cures (returns to performing status). This probability
enters the LGD formula, so besides accuracy we evaluate **ROC-AUC, Gini, Brier score, log-loss and
calibration**.

| Model | Library / device | Tuning |
|---|---|---|
| Baseline (always predicts the training cure rate) | scikit-learn | – |
| Logistic Regression | scikit-learn | – |
| Random Forest | scikit-learn (CPU) | Optuna, 5-fold CV log-loss on train |
| XGBoost | xgboost (CPU) | Optuna, 5-fold CV log-loss on train; boosting rounds by CV early stopping |
| Neural Network (thesis architecture 64-32-16) | **PyTorch on MPS** | early stopping on validation |
| Neural Network (tuned) | **PyTorch on MPS** | Optuna on the **validation** set |
| Ensemble (top 3) | average of the 3 models with the best validation Brier score | – |

The test set is used once, at the end, to report results.

> **Quick mode.** Start Jupyter with `LGD_QUICK=1` to run every model with 3 trials / 3 epochs. That only
> checks that the notebook runs; the results are meaningless. Leave it unset for real results.

In [ ]:
# Reload the lgd package automatically when its files change (no kernel restart needed after edits).
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import optuna
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBClassifier

from lgd import torch_models as tm
from lgd.config import (CV_FOLDS, MAX_EPOCHS, N_TRIALS, N_TRIALS_NN, OUTPUT_DIR, PATIENCE, PREDICTION_DIR,
                        QUICK, SEED)
from lgd.data import feature_columns, load_model_dataset, make_splits
from lgd.metrics import bootstrap_ci, classification_metrics, format_ci, results_table
from lgd.plots import (plot_calibration, plot_coefficients, plot_confusion_matrices, plot_history,
                       plot_optuna_history, plot_roc_curves, plot_shap_summary)
from lgd.tuning import fit_xgboost_with_early_stopping, tune_network, tune_with_cv, tuning_rows

print(f"Quick mode: {QUICK} | PyTorch device: {tm.DEVICE}")

## Data

`FEATURE_SET = "at_default"` uses only what is known when the loan defaults. Set it to `"thesis"` to use every
feature, as in the thesis.

In [ ]:
FEATURE_SET = "at_default"

dataset = load_model_dataset()
data = make_splits(dataset, "cure", FEATURE_SET)
train, valid, test = data["train"], data["valid"], data["test"]
n_features = train.X.shape[1]

# The networks need scaled inputs; the scaler only sees the training rows.
scaler = MinMaxScaler().fit(train.X)
X_train_nn, X_valid_nn = scaler.transform(train.X), scaler.transform(valid.X)
tune_rows = tuning_rows(train.y, classification=True)   # all rows, or a stratified sample for very large data

print(f"{n_features} features | train={len(train)}  valid={len(valid)}  test={len(test)}")
print(f"Cure rate (train): {train.y.mean():.3f} | rows used for hyperparameter searches: {len(tune_rows)}")

In [ ]:
probabilities = {}   # model -> {"valid": P(cure), "test": P(cure)}
test_results = {}    # model -> test-set metrics


def record(model_name: str, predict_proba, train_time: float) -> None:
    """Store validation and test probabilities of a fitted model and print its test metrics."""
    probabilities[model_name] = {"valid": predict_proba(valid.X), "test": predict_proba(test.X)}
    metrics = classification_metrics(test.y, probabilities[model_name]["test"])
    test_results[model_name] = {**metrics, "Training Time (s)": train_time}
    print(pd.Series(test_results[model_name]).round(4).to_string())

# Baseline

In [ ]:
baseline = DummyClassifier(strategy="prior").fit(train.X, train.y)
record("Baseline", lambda X: baseline.predict_proba(X)[:, 1], 0.0)

# Logistic Regression

`liblinear` solver, as in the thesis; the features are Min-Max scaled inside the pipeline.

In [ ]:
logistic_model = make_pipeline(MinMaxScaler(), LogisticRegression(solver="liblinear", max_iter=200))

folds = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
for scoring in ["accuracy", "roc_auc"]:
    scores = cross_val_score(logistic_model, train.X, train.y, cv=folds, scoring=scoring)
    print(f"5-fold CV {scoring}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
start = time.perf_counter()
logistic_model.fit(train.X, train.y)
record("Logistic Regression", lambda X: logistic_model.predict_proba(X)[:, 1], time.perf_counter() - start)

In [ ]:
logistic_coefficients = pd.Series(logistic_model[-1].coef_[0], index=train.X.columns)
plot_coefficients(logistic_coefficients, "Logistic Regression - Largest Coefficients", "cure_logistic_coefficients.png")

# Random Forest

Optuna replaces the grid search: it samples the search space adaptively (TPE) and needs far fewer fits.
Objective: 5-fold CV log-loss on the training set (a proper scoring rule, because we need good probabilities).

In [ ]:
def build_random_forest(trial):
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 400, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.05, 0.6),
        n_jobs=-1, random_state=SEED,
    )


rf_study, rf_model, rf_time = tune_with_cv(build_random_forest, train.X, train.y, scoring="neg_log_loss",
                                           n_trials=N_TRIALS, classification=True, study_name="cure_random_forest")
print("Best parameters:", rf_study.best_params)
plot_optuna_history(rf_study, "Random Forest - Optuna", "−CV log-loss", "cure_random_forest_optuna.png")

In [ ]:
record("Random Forest", lambda X: rf_model.predict_proba(X)[:, 1], rf_time)

# XGBoost

Same procedure as the Random Forest. Afterwards the number of boosting rounds is chosen again by 5-fold
cross-validation with early stopping (the tuned learning rate is kept): at low learning rates the Optuna
range of `n_estimators` is often too small.

In [ ]:
def build_xgboost(trial):
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 1000, step=50),
        max_depth=trial.suggest_int("max_depth", 2, 10),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        eval_metric="logloss", n_jobs=-1, random_state=SEED,
    )


xgb_study, xgb_model, xgb_time = tune_with_cv(build_xgboost, train.X, train.y, scoring="neg_log_loss",
                                              n_trials=N_TRIALS, classification=True, study_name="cure_xgboost")
print("Best parameters:", xgb_study.best_params)
plot_optuna_history(xgb_study, "XGBoost - Optuna", "−CV log-loss", "cure_xgboost_optuna.png")

In [ ]:
xgb_model, xgb_rounds, xgb_time = fit_xgboost_with_early_stopping(xgb_model, train.X, train.y, classification=True)
print(f"Boosting rounds: {xgb_rounds} (CV early stopping) instead of {xgb_study.best_params['n_estimators']} (Optuna)")

record("XGBoost", lambda X: xgb_model.predict_proba(X)[:, 1], xgb_time)

# Neural Network – Thesis Architecture (PyTorch, Apple GPU)

64 → 32 → 16 → 1, ReLU, Adam (lr = 0.003), batch size 64, binary cross-entropy. Instead of a fixed
70 epochs, training stops when the validation loss has not improved for `PATIENCE` epochs and the best
weights are restored.

In [ ]:
tm.set_seed(SEED)
thesis_network = tm.MLP(n_features, hidden=(64, 32, 16), loss="bce")
thesis_history = tm.train_network(thesis_network, X_train_nn, train.y, X_valid_nn, valid.y,
                                  learning_rate=0.003, batch_size=64, max_epochs=MAX_EPOCHS,
                                  patience=PATIENCE, verbose=True)
plot_history(thesis_history, "Neural Network (thesis architecture)", "cure_nn_thesis_training_curves.png")

In [ ]:
record("Neural Network", lambda X: tm.predict(thesis_network, scaler.transform(X)), thesis_history["train_seconds"])

# Neural Network – Tuned with Optuna (PyTorch, Apple GPU)

Searches depth (1–4 hidden layers, each half as wide as the previous one), width, dropout, learning rate,
batch size and weight decay. Every trial is scored on the **validation set** (the thesis used the test set
here); trials that are clearly worse than the median are stopped early.

In [ ]:
def suggest_network(trial):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    width = trial.suggest_int("width", 16, 256, log=True)
    network = tm.MLP(n_features, hidden=tuple(max(4, width // 2 ** i) for i in range(n_layers)),
                     dropout=trial.suggest_float("dropout", 0.0, 0.5), loss="bce")
    fit_options = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256, 512]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }
    return network, fit_options


def network_objective(trial):
    tm.set_seed(SEED)
    network, fit_options = suggest_network(trial)
    history = tm.train_network(network, X_train_nn[tune_rows], train.y.iloc[tune_rows], X_valid_nn, valid.y,
                               max_epochs=MAX_EPOCHS, patience=PATIENCE, trial=trial, **fit_options)
    return history["best_val_loss"]


nn_study = tune_network(network_objective, N_TRIALS_NN, "cure_neural_network")
print("Best parameters:", nn_study.best_params)
plot_optuna_history(nn_study, "Neural Network - Optuna", "Validation log-loss", "cure_nn_optuna.png")

In [ ]:
tm.set_seed(SEED)
tuned_network, fit_options = suggest_network(optuna.trial.FixedTrial(nn_study.best_params))
tuned_history = tm.train_network(tuned_network, X_train_nn, train.y, X_valid_nn, valid.y, max_epochs=MAX_EPOCHS,
                                 patience=PATIENCE, verbose=True, **fit_options)
plot_history(tuned_history, "Neural Network (tuned)", "cure_nn_tuned_training_curves.png")

record("Neural Network (tuned)", lambda X: tm.predict(tuned_network, scaler.transform(X)), tuned_history["train_seconds"])

# Ensemble of the Best Models

The average of the probabilities of the three models with the lowest **validation** Brier score. Averaging
models that make different errors usually gives slightly better and more stable probabilities.

In [ ]:
validation_brier = {name: brier_score_loss(valid.y, p["valid"]) for name, p in probabilities.items() if name != "Baseline"}
ensemble_members = sorted(validation_brier, key=validation_brier.get)[:3]
print("Ensemble members:", ensemble_members)

probabilities["Ensemble (top 3)"] = {split: np.mean([probabilities[name][split] for name in ensemble_members], axis=0)
                                     for split in ["valid", "test"]}
test_results["Ensemble (top 3)"] = {**classification_metrics(test.y, probabilities["Ensemble (top 3)"]["test"]),
                                    "Training Time (s)": sum(test_results[name]["Training Time (s)"] for name in ensemble_members)}
print(pd.Series(test_results["Ensemble (top 3)"]).round(4).to_string())

# Comparison (test set)

95% bootstrap confidence intervals (resampling the test loans) show whether two models really differ.

ROC-AUC, Gini, Brier score, log-loss and the calibration plot matter most, because notebook 04 uses the
probability itself, not only the predicted class.

In [ ]:
test_probabilities = {name: p["test"] for name, p in probabilities.items()}

summary = results_table(test_results)
summary["ROC-AUC 95% CI"] = [format_ci(summary.loc[name, "ROC-AUC"], bootstrap_ci(test.y, p, roc_auc_score))
                             for name, p in test_probabilities.items()]
summary["Brier 95% CI"] = [format_ci(summary.loc[name, "Brier"], bootstrap_ci(test.y, p, brier_score_loss))
                           for name, p in test_probabilities.items()]
summary

In [ ]:
plot_roc_curves(test.y, test_probabilities, "cure_roc_curves.png")
plot_calibration(test.y, {name: p for name, p in test_probabilities.items() if name != "Baseline"}, "cure_calibration.png")
plot_confusion_matrices(test.y, test_probabilities, "cure_confusion_matrices.png")

# Interpretation – SHAP Values (XGBoost)

Each dot is one loan: its position shows how much the feature pushed the predicted log-odds of cure up or
down, its colour the feature value.

In [ ]:
plot_shap_summary(xgb_model, test.X, "XGBoost - SHAP values for P(cure)", "cure_xgboost_shap.png")

# Effect of the Post-Default Features

Logistic Regression and XGBoost (with the tuned parameters) trained on both feature sets.

In [ ]:
if feature_columns(dataset, "thesis") == feature_columns(dataset, "at_default"):
    print("This dataset has no post-default features - nothing to compare.")
else:
    feature_set_data = {name: make_splits(dataset, "cure", name) for name in ["at_default", "thesis"]}
    model_builders = {
        "Logistic Regression": lambda: make_pipeline(MinMaxScaler(), LogisticRegression(solver="liblinear", max_iter=200)),
        "XGBoost": lambda: XGBClassifier(**xgb_study.best_params, eval_metric="logloss", n_jobs=-1, random_state=SEED),
    }

    feature_set_comparison = {}
    for model_name, build in model_builders.items():
        for set_name, splits in feature_set_data.items():
            model = build().fit(splits["train"].X, splits["train"].y)
            proba = model.predict_proba(splits["test"].X)[:, 1]
            feature_set_comparison[(model_name, set_name)] = classification_metrics(splits["test"].y, proba)

    display(pd.DataFrame(feature_set_comparison).T[["Accuracy", "ROC-AUC", "Brier", "Log-loss"]])

# Save Results and Predictions (used by notebook 04)

In [ ]:
validation_results = results_table({name: classification_metrics(valid.y, p["valid"]) for name, p in probabilities.items()})
summary.to_csv(OUTPUT_DIR / "cure_results_test.csv")
validation_results.to_csv(OUTPUT_DIR / "cure_results_valid.csv")

for split_name, split in [("valid", valid), ("test", test)]:
    predictions = pd.DataFrame({name: p[split_name] for name, p in probabilities.items()}, index=split.X.index)
    predictions.to_csv(PREDICTION_DIR / f"cure_{split_name}.csv")

validation_results